# GuidaPlate — Next-Meal Risk Prediction v2 (Sequential Improvements)## Notebook 10**Companion to nb09** — same task and data, but with four sequential improvements evaluated via **5-fold patient-level cross-validation** (not overwriting nb09).| Step | Change ||------|--------|| 1 | CV baseline — exact nb09 GRU architecture, no modifications || 2 | + balanced class weights in loss || 3 | + one-hot `meal_risk_state` per prefix slot (keep class weights) || 4 | Architecture sweep (4 configs, same folds) |**Outputs:** `outputs/stats/18_trend_prediction_v2_comparison.csv`, `models/trend_gru_v2.keras` (+ scaler/meta). **Not integrated into backend.**

In [ ]:
import os
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from pathlib import Path
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, confusion_matrix, f1_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Concatenate, Dense, Dropout, GRU, Input
from tensorflow.keras.models import Model
from tensorflow.keras.utils import to_categorical

try:
    plt.style.use('seaborn-v0_8')
except OSError:
    plt.style.use('seaborn')

RANDOM_STATE = 42
N_SPLITS = 5
N_STEPS = 6
N_NUTRIENTS = 4
N_RISK_OH = 3
RISK_CLASSES = ['LOW', 'MODERATE', 'HIGH']
RISK_ENCODE = {c: i for i, c in enumerate(RISK_CLASSES)}
T = {'potassium': 3500/3, 'phosphorus': 1000/3, 'protein': 0.8/3, 'sodium': 2300/3}

def project_root():
    p = Path('.').resolve()
    if p.name == 'notebooks':
        return p.parent
    if (p / 'models').exists():
        return p
    return p.parent

ROOT = project_root()
MODEL_DIR = ROOT / 'models'
STATS_DIR = ROOT / 'outputs' / 'stats'
FIG_DIR = ROOT / 'outputs' / 'figures'
STATS_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
print(f'Project root: {ROOT}')


## Shared utilities — examples, model, CVSame example-generation rules as nb09: exclude empty target/prefix slots; patient-level folds via `StratifiedKFold(n_splits=5, random_state=42)` stratified by full-day label.

In [ ]:
def is_empty_slot(vec):
    return not np.any(vec != 0)

def meal_risk_state(meal_vec):
    k, p, pr, s = meal_vec
    ex = sum([k > T['potassium'], p > T['phosphorus'], pr > T['protein'], s > T['sodium']])
    return 2 if ex >= 2 else 1 if ex == 1 else 0

def risk_onehot(state):
    oh = np.zeros(N_RISK_OH)
    oh[state] = 1.0
    return oh

def build_examples(sequences, patient_ids):
    X_nut, X_risk, lens, ys, pids = [], [], [], [], []
    ex_t = ex_i = 0
    for i in range(len(sequences)):
        seq = sequences[i]
        states = [meal_risk_state(seq[j]) for j in range(N_STEPS)]
        for t in range(N_STEPS - 1):
            if is_empty_slot(seq[t + 1]):
                ex_t += 1; continue
            if any(is_empty_slot(seq[k]) for k in range(t + 1)):
                ex_i += 1; continue
            nut = np.zeros((N_STEPS, N_NUTRIENTS))
            nut[:t+1] = seq[:t+1]
            risk = np.zeros((N_STEPS, N_RISK_OH))
            for s in range(t + 1):
                risk[s] = risk_onehot(states[s])
            X_nut.append(nut); X_risk.append(risk)
            lens.append(t + 1); ys.append(states[t + 1]); pids.append(patient_ids[i])
    return {'X_nut': np.array(X_nut), 'X_risk': np.array(X_risk), 'seq_len': np.array(lens, dtype=float),
            'y': np.array(ys), 'patient_ids': np.array(pids), 'excluded_target': ex_t, 'excluded_input': ex_i}

def scale_nutrients(X_nut, scaler):
    n = X_nut.shape[0]
    return scaler.transform(X_nut.reshape(-1, N_NUTRIENTS)).reshape(n, N_STEPS, N_NUTRIENTS)

def featurize(X_nut, X_risk, scaler, use_risk):
    nut_s = scale_nutrients(X_nut, scaler)
    return np.concatenate([nut_s, X_risk], axis=-1) if use_risk else nut_s

def build_model(n_features, units, layers, dropout):
    seq_in = Input(shape=(N_STEPS, n_features))
    len_in = Input(shape=(1,))
    x = seq_in
    for i in range(layers):
        x = GRU(units, dropout=dropout, return_sequences=(i < layers - 1), name=f'gru_{i+1}')(x)
    x = Concatenate()([x, len_in])
    x = Dense(16, activation='relu')(x)
    x = Dropout(dropout)(x)
    out = Dense(3, activation='softmax')(x)
    model = Model([seq_in, len_in], out)
    model.compile(optimizer=tf.keras.optimizers.Adam(0.001), loss='categorical_crossentropy', metrics=['accuracy'])
    return model

def compute_metrics(y_true, y_pred, y_proba):
    f1p = f1_score(y_true, y_pred, average=None, labels=[0,1,2], zero_division=0)
    hi = RISK_ENCODE['HIGH']
    hs = ((y_true == hi) & (y_pred == hi)).sum() / max((y_true == hi).sum(), 1)
    return {'accuracy': accuracy_score(y_true, y_pred), 'f1_weighted': f1_score(y_true, y_pred, average='weighted', zero_division=0),
            'f1_LOW': float(f1p[0]), 'f1_MODERATE': float(f1p[1]), 'f1_HIGH': float(f1p[2]),
            'high_sensitivity': float(hs), 'auc_roc': float(roc_auc_score(y_true, y_proba, multi_class='ovr', average='weighted'))}

def patient_mask(pids, selected):
    selected = set(selected)
    return np.array([p in selected for p in pids])

def train_eval(X_nut_tr, X_risk_tr, len_tr, y_tr, X_nut_va, X_risk_va, len_va, y_va, cfg, use_risk, class_weight):
    tf.random.set_seed(RANDOM_STATE)
    scaler = StandardScaler()
    scaler.fit(X_nut_tr.reshape(-1, N_NUTRIENTS))
    X_tr = featurize(X_nut_tr, X_risk_tr, scaler, use_risk)
    X_va = featurize(X_nut_va, X_risk_va, scaler, use_risk)
    len_tr_n = (len_tr - 1) / (N_STEPS - 1)
    len_va_n = (len_va - 1) / (N_STEPS - 1)
    cw = None
    if class_weight:
        w = compute_class_weight('balanced', classes=np.array([0,1,2]), y=y_tr)
        cw = {i: float(w[i]) for i in range(3)}
    model = build_model(X_tr.shape[2], cfg['units'], cfg['layers'], cfg['dropout'])
    history = model.fit([X_tr, len_tr_n], to_categorical(y_tr, 3),
        validation_data=([X_va, len_va_n], to_categorical(y_va, 3)),
        epochs=50, batch_size=64, class_weight=cw, verbose=0,
        callbacks=[EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True, verbose=0)])
    proba = model.predict([X_va, len_va_n], verbose=0)
    pred = np.argmax(proba, axis=1)
    return compute_metrics(y_va, pred, proba), model, scaler, history

def run_cv(data, patient_folds, cfg, use_risk, class_weight):
    folds = []
    for train_pids, val_pids in patient_folds:
        tr = patient_mask(data['patient_ids'], train_pids)
        va = patient_mask(data['patient_ids'], val_pids)
        m, _, _, _ = train_eval(data['X_nut'][tr], data['X_risk'][tr], data['seq_len'][tr], data['y'][tr],
            data['X_nut'][va], data['X_risk'][va], data['seq_len'][va], data['y'][va], cfg, use_risk, class_weight)
        folds.append(m)
    return folds

def summarize(folds):
    df = pd.DataFrame(folds)
    means = {c: df[c].mean() for c in df.columns}
    stds = {f'std_{c}': df[c].std() for c in df.columns}
    return means, stds, df


In [ ]:
cache = np.load(MODEL_DIR / 'lstm_sequences_cache.npz', allow_pickle=True)
sequences = cache['sequences']
full_labels = cache['labels']
patient_ids_seq = cache['patient_ids']
data = build_examples(sequences, patient_ids_seq)

print(f'Examples: {len(data["y"]):,} (same as nb09)')
print(f'Excluded target: {data["excluded_target"]:,} | Excluded prefix: {data["excluded_input"]:,}')

unique_pids = np.unique(patient_ids_seq)
pid_label = [RISK_ENCODE[full_labels[np.where(patient_ids_seq == p)[0][0]]] for p in unique_pids]
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
patient_folds = [(unique_pids[tr], unique_pids[va]) for tr, va in skf.split(unique_pids, pid_label)]
base_cfg = {'units': 32, 'layers': 1, 'dropout': 0.2, 'label': 'baseline (current)'}


## Step 1 — 5-fold CV baseline (nb09 architecture)GRU(32, dropout=0.2) + seq_length + Dense(16) → softmax(3). **No class weights, nutrients only.**

In [ ]:
print('Running Step 1 (5-fold CV)...')
s1_folds = run_cv(data, patient_folds, base_cfg, use_risk=False, class_weight=False)
s1_mean, s1_std, s1_df = summarize(s1_folds)
print('\n=== STEP 1 — CV BASELINE (mean ± std) ===')
for k in s1_mean:
    print(f'  {k}: {s1_mean[k]:.4f} ± {s1_std[f"std_{k}"]:.4f}')
display(s1_df.round(4))


## Step 2 — + balanced class weightsSame architecture/data as Step 1; `compute_class_weight('balanced')` applied each fold.

In [ ]:
print('Running Step 2...')
s2_folds = run_cv(data, patient_folds, base_cfg, use_risk=False, class_weight=True)
s2_mean, s2_std, s2_df = summarize(s2_folds)

comp12 = pd.DataFrame([{'metric': k, 'step1': round(s1_mean[k],4), 'step2': round(s2_mean[k],4),
    'delta': round(s2_mean[k]-s1_mean[k],4)} for k in s1_mean])
print('\n=== STEP 1 vs STEP 2 ===')
display(comp12)
comp12.to_csv(STATS_DIR / '18_trend_v2_step1_vs_step2.csv', index=False)


## Step 3 — + one-hot meal risk state per slotAppend 3-dim one-hot risk label for each real prefix slot (`[0,0,0]` for padded). Keep class weights from Step 2.

In [ ]:
print('Running Step 3...')
s3_folds = run_cv(data, patient_folds, base_cfg, use_risk=True, class_weight=True)
s3_mean, s3_std, s3_df = summarize(s3_folds)

comp23 = pd.DataFrame([{'metric': k, 'step2': round(s2_mean[k],4), 'step3': round(s3_mean[k],4),
    'delta': round(s3_mean[k]-s2_mean[k],4)} for k in s2_mean])
print('\n=== STEP 2 vs STEP 3 ===')
display(comp23)
comp23.to_csv(STATS_DIR / '18_trend_v2_step2_vs_step3.csv', index=False)


## Step 4 — Architecture sweepBest config so far (class weights + risk features). **Same fold splits** for all 4 configs. Winner = highest mean weighted F1.

In [ ]:
configs = [
    {'units': 32, 'layers': 1, 'dropout': 0.2, 'label': 'baseline (current)'},
    {'units': 64, 'layers': 1, 'dropout': 0.2, 'label': 'larger single layer'},
    {'units': 32, 'layers': 1, 'dropout': 0.4, 'label': 'higher dropout'},
    {'units': 32, 'layers': 2, 'dropout': 0.3, 'label': 'stacked GRU'},
]
step4_rows = []
for cfg in configs:
    print(f'  CV: {cfg["label"]}...')
    fm = run_cv(data, patient_folds, cfg, use_risk=True, class_weight=True)
    m, s, _ = summarize(fm)
    row = {'config': cfg['label']}
    for k in m:
        row[k] = round(m[k], 4); row[f'std_{k}'] = round(s[f'std_{k}'], 4)
    step4_rows.append(row)

step4_df = pd.DataFrame(step4_rows)
winner_cfg = configs[int(step4_df['f1_weighted'].idxmax())]
print(f'\nWinner (weighted F1): {winner_cfg["label"]}')
display(step4_df)
step4_df.to_csv(STATS_DIR / '18_trend_prediction_v2_arch_sweep.csv', index=False)


## Final — Progression summary & artifact exportTrain winning config on nb09-style 80/20 patient split for held-out confusion matrix and save model artifacts.

In [ ]:
nb09 = pd.read_csv(STATS_DIR / '17_trend_prediction_comparison.csv')
r = nb09[nb09['model'] == 'trained_GRU'].iloc[0]
progression = [
    {'stage': 'nb09_single_split', 'accuracy': r['accuracy'], 'f1_weighted': r['f1_weighted'],
     'f1_MODERATE': r['f1_MODERATE'], 'high_sensitivity': r['high_sensitivity'], 'auc_roc': r['auc_roc']},
    {'stage': 'step1_cv_baseline', **{k: round(s1_mean[k],4) for k in s1_mean},
     **{f'std_{k}': round(s1_std[f'std_{k}'],4) for k in s1_mean}},
    {'stage': 'step2_class_weights', **{k: round(s2_mean[k],4) for k in s2_mean},
     **{f'std_{k}': round(s2_std[f'std_{k}'],4) for k in s2_mean}},
    {'stage': 'step3_risk_features', **{k: round(s3_mean[k],4) for k in s3_mean},
     **{f'std_{k}': round(s3_std[f'std_{k}'],4) for k in s3_mean}},
]
w = step4_df[step4_df['config'] == winner_cfg['label']].iloc[0]
progression.append({'stage': f'step4_{winner_cfg["label"]}', 'accuracy': w['accuracy'],
    'f1_weighted': w['f1_weighted'], 'f1_MODERATE': w['f1_MODERATE'],
    'high_sensitivity': w['high_sensitivity'], 'auc_roc': w['auc_roc'],
    'std_accuracy': w['std_accuracy'], 'std_f1_weighted': w['std_f1_weighted']})
prog_df = pd.DataFrame(progression)
prog_df.to_csv(STATS_DIR / '18_trend_prediction_v2_comparison.csv', index=False)

print('=' * 60)
print('FULL PROGRESSION')
print('=' * 60)
display(prog_df[['stage','accuracy','f1_weighted','f1_MODERATE','high_sensitivity','auc_roc']])

# Final 80/20 model
train_pids, test_pids = train_test_split(unique_pids, test_size=0.2, random_state=RANDOM_STATE, stratify=pid_label)
tr = patient_mask(data['patient_ids'], train_pids)
va = patient_mask(data['patient_ids'], test_pids)
metrics, model, scaler, history = train_eval(
    data['X_nut'][tr], data['X_risk'][tr], data['seq_len'][tr], data['y'][tr],
    data['X_nut'][va], data['X_risk'][va], data['seq_len'][va], data['y'][va],
    winner_cfg, use_risk=True, class_weight=True)
model.save(MODEL_DIR / 'trend_gru_v2.keras')
joblib.dump(scaler, MODEL_DIR / 'trend_gru_v2_scaler.pkl')
joblib.dump({'config': winner_cfg, 'use_risk': True, 'class_weight': True}, MODEL_DIR / 'trend_gru_v2_meta.pkl')

X_va = featurize(data['X_nut'][va], data['X_risk'][va], scaler, True)
pred = np.argmax(model.predict([X_va, (data['seq_len'][va]-1)/(N_STEPS-1)], verbose=0), axis=1)
cm = confusion_matrix(data['y'][va], pred, labels=[0,1,2])

fig, ax = plt.subplots(figsize=(7,6))
ConfusionMatrixDisplay(cm, display_labels=RISK_CLASSES).plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title(f'Trend GRU v2 — {winner_cfg["label"]} (80/20 test)')
plt.tight_layout(); plt.savefig(FIG_DIR / '28_trend_prediction_v2_confusion_matrix.png', dpi=150); plt.show()

fig, axes = plt.subplots(1,2,figsize=(12,4))
axes[0].plot(history.history['accuracy'], label='Train'); axes[0].plot(history.history['val_accuracy'], label='Val')
axes[0].set_title('Training Accuracy'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].plot(history.history['loss'], label='Train'); axes[1].plot(history.history['val_loss'], label='Val')
axes[1].set_title('Training Loss'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig(FIG_DIR / '29_trend_prediction_v2_training_history.png', dpi=150); plt.show()

print('\n80/20 held-out test metrics:')
for k,v in metrics.items():
    print(f'  {k}: {v:.4f}')


## Interpretation**Key findings (honest):**1. **Step 1 CV baseline** (~61.7% accuracy) confirms nb09's single-split 61% was **representative**, not a lucky draw. MODERATE F1 remains **0.00** across all 5 folds without class weighting — the model never predicts MODERATE.2. **Class weighting (Step 2)** is the first intervention that produces non-zero MODERATE F1 (~0.16), but at a **steep cost**: accuracy drops ~15 pp, HIGH F1 drops ~0.20, HIGH sensitivity falls from 0.93 → 0.50. This is the classic majority-vs-minority tradeoff.3. **Risk-state features (Step 3)** recover accuracy (+7 pp vs Step 2) while keeping MODERATE F1 elevated (~0.18). AUC and HIGH sensitivity improve vs Step 2 but remain below the unweighted baseline.4. **Architecture sweep (Step 4):** Stacked GRU wins on weighted F1 (0.545) but **MODERATE F1 drops** vs Step 3 (0.145 vs 0.185). No configuration meaningfully solves MODERATE prediction — the best MODERATE F1 in the sweep is ~0.19 (baseline config with risk features), still very weak for a 13% prevalence class.**Conclusion:** Next-meal MODERATE state prediction remains fundamentally difficult. v2 improvements lift weighted F1 modestly (~0.53 → 0.55 CV) and enable occasional MODERATE predictions, but none of the four interventions produce a clinically reliable MODERATE detector.